## Notebook 03: Key Insights & KPI Extraction
### Volkswagen Stock Data 2000–2025

This notebook extracts specific numbers from the cleaned dataset that will 
become KPI cards and talking points in the Power BI dashboard.

Input: data/processed/vw_stock_features.csv

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\vw_project\data\processed\vw_stock_features.csv',
                 index_col='Date',
                 parse_dates=True)

print('Data loaded:', df.shape)
print('Date range:', df.index.min().date(), 'to', df.index.max().date())

Data loaded: (6684, 25)
Date range: 2000-01-03 to 2026-02-26


In [2]:
print('===== VW STOCK: CORE KPIs =====')
print(f'All-Time High:        €{df["High"].max():.2f}')
print(f'All-Time Low:         €{df["Low"].min():.2f}')
print(f'Latest Close Price:   €{df["Close"].iloc[-1]:.2f}')
print(f'Total Trading Days:   {len(df):,}')
print(f'Average Daily Volume: {df["Volume"].mean():,.0f} shares')
print(f'Average Daily Return: {df["Daily_Return_Pct"].mean():.4f}%')

===== VW STOCK: CORE KPIs =====
All-Time High:        €635.00
All-Time Low:         €28.06
Latest Close Price:   €101.80
Total Trading Days:   6,684
Average Daily Volume: 921,909 shares
Average Daily Return: 0.0435%


In [3]:
annual = df.groupby('Year')['Close'].agg(['first', 'last'])
annual['Return_Pct'] = ((annual['last'] - annual['first']) / annual['first'] * 100).round(2)

print('===== BEST AND WORST YEARS =====')
print(f'Best Year:  {annual["Return_Pct"].idxmax()} at {annual["Return_Pct"].max():.1f}%')
print(f'Worst Year: {annual["Return_Pct"].idxmin()} at {annual["Return_Pct"].min():.1f}%')

===== BEST AND WORST YEARS =====
Best Year:  2006 at 90.4%
Worst Year: 2009 at -70.3%


In [4]:
print('===== CRASH EVENT ANALYSIS =====')

# Dieselgate
peak_pre_diesel = df['2015-01-01':'2015-09-17']['Close'].max()
trough_diesel   = df['2015-09-18':'2015-12-31']['Close'].min()
diesel_crash    = (trough_diesel - peak_pre_diesel) / peak_pre_diesel * 100
print(f'Dieselgate Crash:  {diesel_crash:.1f}% from peak to trough')

# COVID
peak_pre_covid = df['2020-01-01':'2020-03-15']['Close'].max()
trough_covid   = df['2020-03-16':'2020-05-01']['Close'].min()
covid_crash    = (trough_covid - peak_pre_covid) / peak_pre_covid * 100
print(f'COVID-19 Crash:    {covid_crash:.1f}% from peak to trough')

# 2008
peak_pre_2008 = df['2007-01-01':'2008-09-14']['Close'].max()
trough_2008   = df['2008-09-15':'2009-06-01']['Close'].min()
crisis_crash  = (trough_2008 - peak_pre_2008) / peak_pre_2008 * 100
print(f'2008 Crisis Crash: {crisis_crash:.1f}% from peak to trough')

===== CRASH EVENT ANALYSIS =====
Dieselgate Crash:  -59.1% from peak to trough
COVID-19 Crash:    -44.6% from peak to trough
2008 Crisis Crash: -12.1% from peak to trough


In [5]:
print('===== EXTREME SINGLE DAY MOVES =====')

extreme = df[df['Daily_Return_Pct'].abs() > 5]
print(f'Days with move greater than 5%: {len(extreme)}')

print('\nTop 5 Best Single Days:')
print(df['Daily_Return_Pct'].nlargest(5).round(2))

print('\nTop 5 Worst Single Days:')
print(df['Daily_Return_Pct'].nsmallest(5).round(2))

===== EXTREME SINGLE DAY MOVES =====
Days with move greater than 5%: 241

Top 5 Best Single Days:
Date
2008-10-27    146.62
2008-09-18     26.57
2021-03-17     15.83
2008-11-26     15.33
2008-10-10     15.20
Name: Daily_Return_Pct, dtype: float64

Top 5 Worst Single Days:
Date
2008-11-25   -22.66
2008-10-20   -22.60
2008-11-03   -21.32
2015-09-21   -17.14
2015-09-22   -16.83
Name: Daily_Return_Pct, dtype: float64


In [6]:
print('===== MAXIMUM DRAWDOWN =====')

rolling_max = df['Close'].cummax()
drawdown    = (df['Close'] - rolling_max) / rolling_max * 100

max_drawdown      = drawdown.min()
max_drawdown_date = drawdown.idxmin()

print(f'Maximum Drawdown:      {max_drawdown:.1f}%')
print(f'Date of Worst Trough:  {max_drawdown_date.date()}')

===== MAXIMUM DRAWDOWN =====
Maximum Drawdown:      -88.0%
Date of Worst Trough:  2010-02-12


In [7]:
print('===== AVERAGE RETURN BY MONTH =====')

monthly_avg = df.groupby('Month')['Daily_Return_Pct'].mean().round(4)
monthly_avg.index = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec']

print(monthly_avg.sort_values(ascending=False))
print(f'\nStrongest Month: {monthly_avg.idxmax()}')
print(f'Weakest Month:   {monthly_avg.idxmin()}')

===== AVERAGE RETURN BY MONTH =====
Oct    0.4848
Jul    0.1580
Mar    0.0961
Jan    0.0951
Apr    0.0444
Feb    0.0277
Dec   -0.0169
Nov   -0.0295
May   -0.0446
Jun   -0.0833
Sep   -0.0863
Aug   -0.1393
Name: Daily_Return_Pct, dtype: float64

Strongest Month: Oct
Weakest Month:   Aug


In [8]:
print('===== RSI SUMMARY =====')

overbought = df[df['RSI_14'] > 70]
oversold   = df[df['RSI_14'] < 30]

print(f'Days overbought (RSI above 70): {len(overbought)} ({len(overbought)/len(df)*100:.1f}% of all days)')
print(f'Days oversold (RSI below 30):   {len(oversold)} ({len(oversold)/len(df)*100:.1f}% of all days)')
print(f'Highest RSI recorded: {df["RSI_14"].max():.1f} on {df["RSI_14"].idxmax().date()}')
print(f'Lowest RSI recorded:  {df["RSI_14"].min():.1f} on {df["RSI_14"].idxmin().date()}')

===== RSI SUMMARY =====
Days overbought (RSI above 70): 1081 (16.2% of all days)
Days oversold (RSI below 30):   857 (12.8% of all days)
Highest RSI recorded: 99.2 on 2017-09-19
Lowest RSI recorded:  1.6 on 2023-08-14


In [9]:
insights = {
    'All_Time_High':        df['High'].max(),
    'All_Time_Low':         df['Low'].min(),
    'Latest_Close':         df['Close'].iloc[-1],
    'Total_Trading_Days':   len(df),
    'Avg_Daily_Volume':     df['Volume'].mean(),
    'Avg_Daily_Return':     df['Daily_Return_Pct'].mean(),
    'Best_Year':            int(annual['Return_Pct'].idxmax()),
    'Best_Year_Return':     annual['Return_Pct'].max(),
    'Worst_Year':           int(annual['Return_Pct'].idxmin()),
    'Worst_Year_Return':    annual['Return_Pct'].min(),
    'Max_Drawdown':         drawdown.min(),
    'Days_Overbought':      len(overbought),
    'Days_Oversold':        len(oversold),
}

insights_df = pd.DataFrame(list(insights.items()), columns=['Metric', 'Value'])
insights_df['Value'] = insights_df['Value'].round(2)
insights_df.to_csv(r'C:\vw_project\data\processed\vw_key_insights.csv', index=False)

print('All insights saved successfully.')
print(insights_df.to_string(index=False))

All insights saved successfully.
            Metric     Value
     All_Time_High    635.00
      All_Time_Low     28.06
      Latest_Close    101.80
Total_Trading_Days   6684.00
  Avg_Daily_Volume 921909.36
  Avg_Daily_Return      0.04
         Best_Year   2006.00
  Best_Year_Return     90.44
        Worst_Year   2009.00
 Worst_Year_Return    -70.28
      Max_Drawdown    -88.02
   Days_Overbought   1081.00
     Days_Oversold    857.00


## Insights Summary

All key metrics have been extracted and saved to:
data/processed/vw_key_insights.csv

These numbers will be used as KPI cards in the Power BI dashboard.